In [1]:
import numpy as np

from obspy import read, Stream, Trace
import matplotlib.pyplot as plt

%matplotlib qt

In [2]:
# file_path = r"D:\PCCP\FIPdata1\phase_data_20260320_203733_046_#002249.npz"

file_path = r"E:\codes\pccpHOST\wb-monitor\read\0000003-FIP-200K-20260322T050356.689.npz"

with np.load(file_path, allow_pickle=True) as data:
    data_info = data["data_info"]
    phase_data = np.asarray(data["phase_data"], dtype=np.float64)

tr = Trace()
tr.data = phase_data
tr.stats.sampling_rate = 200000
# tr.plot(method= 'full')

tr.filter('highpass', freq=30000)
tr.taper(type='hann', max_percentage=0.05)
fig = plt.figure(figsize=(10, 4))
tr.plot(method= 'full', fig=fig)
plt.ylabel('Phase (rad)')
plt.title('AE Data')

Text(0.5, 1.0, 'AE Data')

In [3]:
# welch 计算PSD，并绘制图形
from scipy.signal import welch
fs = tr.stats.sampling_rate  # 采样率
tr_ae2 = tr.copy().trim(starttime=tr.stats.starttime+4.5, endtime=tr.stats.starttime + 4.6)  # 取前1秒数据计算PSD
tr_noise2 = tr.copy().trim(starttime=tr.stats.starttime+4.0, endtime=tr.stats.starttime + 4.1)  # 取前0.5秒数据计算PSD
tr_ae2.plot(method='full', fig=plt.figure(figsize=(10, 4)))
f_ae2, Pxx_ae2 = welch(tr_ae2.data, fs=fs, nperseg=10240)
f_noise2, Pxx_noise2 = welch(tr_noise2.data, fs=fs, nperseg=10240)
plt.figure(figsize=(10, 6))
plt.plot(f_ae2, Pxx_ae2)    #两个轴全对数
plt.xscale('log'); plt.yscale('log')
plt.plot(f_noise2, Pxx_noise2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD (rad^2/Hz)')
plt.grid(True, which='both', ls='--')
plt.xlim(100, fs/2)
plt.title('Power Spectral Density (Welch Method)')

Text(0.5, 1.0, 'Power Spectral Density (Welch Method)')

## DAS